In [2]:
import json
import os
import copy

# 入出力パス（必要なら変更）
INPUT_JSON = r"E:\water.json"
OUTPUT_DIR  = r"E:\methanewater"

# 出力先フォルダを作成
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 元データを読み込み
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    base_data = json.load(f)

# H2OSTR を 0.00 ～ 7.50（0.25刻み）でループ
start, stop, step = 0.00, 7.50, 0.25
n_steps = int(round((stop - start) / step)) + 1

for i in range(n_steps):
    h2o = round(start + i * step, 2)  # 丸めて表記ブレ対策

    # 元データのディープコピーを作成して編集
    data = copy.deepcopy(base_data)

    # 対象フィールドを書き換え
    modtran_input = data["MODTRAN"][0]["MODTRANINPUT"]
    modtran_input["ATMOSPHERE"]["H2OSTR"] = h2o

    # CSVPRNT を値ごとに分かりやすい名前へ
    # 例: ch_h2ostr_0.00.csv, ch_h2ostr_0.25.csv, ...
    csv_name = f"ch_h2ostr_{h2o:0.2f}.csv"
    modtran_input["FILEOPTIONS"]["CSVPRNT"] = csv_name

    # JSON 自体の出力ファイル名も分かりやすく
    # 例: water_H2OSTR_0.00.json など
    out_json = os.path.join(OUTPUT_DIR, f"water_H2OSTR_{h2o:0.2f}.json")

    # 書き出し（UTF-8・整形）
    with open(out_json, "w", encoding="utf-8") as fout:
        json.dump(data, fout, ensure_ascii=False, indent=2)

print(f"Done. {n_steps} files written to {OUTPUT_DIR}")


Done. 31 files written to E:\methanewater


In [1]:
import json
import os
import copy

# 入出力パス
INPUT_JSON = r"E:\methanewide\sample.json"
OUTPUT_DIR  = r"E:\methanewide\bgn"

# 出力先フォルダを作成
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 元データを読み込み
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    base_data = json.load(f)

# SURREF を 0.00 ～ 0.80（0.01刻み）でループ
start, stop, step = 0.00, 0.80, 0.01
n_steps = int(round((stop - start) / step)) + 1  # 81通り

for i in range(n_steps):
    surref = round(start + i * step, 2)  # 表記ブレ対策で小数点2桁に丸め

    # ディープコピーを作って編集
    data = copy.deepcopy(base_data)

    # JSONの該当箇所を更新
    modtran_input = data["MODTRAN"][0]["MODTRANINPUT"]
    modtran_input["SURFACE"]["SURREF"] = surref

    # CSVPRNT も「いい感じの名前」に変更（例: bg_surref_0.37.csv）
    csv_name = f"bg_surref_{surref:0.2f}.csv"
    modtran_input["FILEOPTIONS"]["CSVPRNT"] = csv_name

    # 出力JSON名（例: sample_SURREF_0.37.json）
    out_json = os.path.join(OUTPUT_DIR, f"{surref:0.2f}.json")

    # JSONを書き出し（整形・UTF-8）
    with open(out_json, "w", encoding="utf-8") as fout:
        json.dump(data, fout, ensure_ascii=False, indent=2)

print(f"Done. {n_steps} files written to {OUTPUT_DIR}")


Done. 81 files written to E:\methanewide\bgn


In [2]:
import json
import os
import copy

# 入出力パス
INPUT_JSON = r"E:\methanewide\sample.json"
OUTPUT_DIR = r"E:\methanewide\testn"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 元データ読み込み
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    base_data = json.load(f)

# SURREF の範囲（0.00〜0.80, 0.01刻み）
start, stop, step = 0.00, 0.80, 0.01
n_surref = int(round((stop - start) / step)) + 1  # 81

# 先頭11要素（0.0〜10.0 km）を一定値に置換する候補（13通り）
# 1.8 + [0.1,0.3,0.5,1.0,2.0,3.0,5.0,7.5,10.0,15.0,20.0,25.0,30.0]
head_values = [1.8 + d for d in [0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0, 15.0, 20.0, 25.0, 30.0]]
BACKGROUND_VALUE = 1.8          # 12個目以降（>10.0 km）は 1.8 で埋める
HEAD_LEN = 11                   # 0.0〜10.0 km の11要素

# 小数表記の安全化（例: 0.300000000004 → 0.30）
def f2(x):
    return float(f"{x:.2f}")

# すべての SURREF × 13 パターンを生成
count = 0
for i in range(n_surref):
    surref = f2(start + i * step)

    for v in head_values:
        v = f2(v)

        # ディープコピーして編集
        data = copy.deepcopy(base_data)
        modtran_input = data["MODTRAN"][0]["MODTRANINPUT"]

        # SURREF を更新
        modtran_input["SURFACE"]["SURREF"] = surref

        # PROF_CH4 / UNT_DPPMV を探して PROFILE を編集
        # ・先頭11要素を v に
        # ・それ以降は 1.8 に
        profiles = modtran_input["ATMOSPHERE"]["PROFILES"]
        edited_any = False
        for prof in profiles:
            if prof.get("TYPE") == "PROF_CH4" and prof.get("UNITS") == "UNT_DPPMV":
                arr = prof.get("PROFILE", [])
                if len(arr) < HEAD_LEN:
                    raise ValueError(f"PROFILE の長さが {len(arr)} しかありません（最低 {HEAD_LEN} 必要）")

                # 先頭11要素を v、残りは 1.8 で埋める
                new_profile = [v] * HEAD_LEN + [BACKGROUND_VALUE] * max(0, len(arr) - HEAD_LEN)
                prof["PROFILE"] = new_profile
                edited_any = True

        if not edited_any:
            raise KeyError('ATMOSPHERE.PROFILES 内に TYPE="PROF_CH4" かつ UNITS="UNT_DPPMV" の要素が見つかりません。')

        # CSVPRNT をわかりやすい名前に（例: test_surref_0.37_ch4_2.10.csv）
        csv_name = f"test_surref_{surref:0.2f}_ch4_{v:0.2f}.csv"
        modtran_input["FILEOPTIONS"]["CSVPRNT"] = csv_name

        # 出力ファイル名（例: sample_SURREF_0.37_ch4_2.10.json）
        out_json = os.path.join(OUTPUT_DIR, f"{surref:0.2f}_ch4_{v:0.2f}.json")

        with open(out_json, "w", encoding="utf-8") as fout:
            json.dump(data, fout, ensure_ascii=False, indent=2)

        count += 1

print(f"Done. {count} files written to {OUTPUT_DIR}")


Done. 1053 files written to E:\methanewide\testn
